# 🛡️ IDS Lab with Suricata — Offline PCAP Analysis
## Running on VirtualBox/Vagrant · Local Environment

---

**Practice objective:**

Running this notebook will let you:
- Verify and interpret the Suricata configuration (`suricata.yaml`)
- Update rules with `suricata-update` and justify the impact
- Analyze two PCAPs **offline** with different threat profiles
- Compare results between web scans and WannaCry/EternalBlue traffic
- Write a technical report based on real evidence

**Steps to run (in order):**
1. ✅ Environment check and Suricata installation
2. ✅ Detailed review of `/etc/suricata/suricata.yaml`
3. ✅ Rule update with `suricata-update`
4. ✅ Offline analysis — PCAP 1: *webserver scans and probes*
5. ✅ Offline analysis — PCAP 2: *WannaCry / EternalBlue*
6. ✅ Comparative analysis and configuration proposals
7. ✅ Technical report template

> ⚠️ **Important:** Run the cells **in order**. If your environment doesn't support live capture, focus on the offline PCAP mode (this is the main objective of the practice).

---
## Section 1: Environment Check and Suricata Installation

This section installs Suricata (if not already available), dynamically detects the execution environment, and validates that all required components are operational.

In [ ]:
import subprocess
import os
import sys
import glob
import json
import shutil
from pathlib import Path

# Helper function to read protected files
def read_file_safe(filepath: Path) -> str:
    """Reads a file, handling PermissionError with sudo."""
    try:
        return filepath.read_text()
    except PermissionError:
        result = subprocess.run(['sudo', 'cat', str(filepath)],
                              capture_output=True, text=True)
        if result.returncode == 0:
            return result.stdout
        else:
            return ""
    except FileNotFoundError:
        return ""

# Detect execution environment
def detect_environment():
    """Detects whether we're on Colab, Vagrant/VirtualBox, or a generic environment."""
    if 'google.colab' in sys.modules or os.path.exists('/content'):
        return 'colab', Path('/content')
    if os.path.exists('/home/vagrant'):
        return 'vagrant', Path('/home/vagrant')
    return 'generic', Path.home()

ENV_TYPE, WORK_DIR = detect_environment()
PCAP_DIR    = WORK_DIR / 'pcaps'
RESULTS_DIR = WORK_DIR / 'suricata_results'
LOG_DIR     = Path('/var/log/suricata')
RULES_DIR   = Path('/var/lib/suricata/rules')
CONFIG_FILE = Path('/etc/suricata/suricata.yaml')

PCAP_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Detected environment:", ENV_TYPE)
print("Working directory:", WORK_DIR)
print("PCAPs directory:", PCAP_DIR)
print("Results directory:", RESULTS_DIR)
print("Suricata configuration:", CONFIG_FILE)

In [ ]:
result = subprocess.run(['suricata', '--build-info'], capture_output=True, text=True)
for line in result.stdout.splitlines():
    if 'Version' in line or 'version' in line:
        print(line.strip())

# Check sudo permissions
sudo_check = subprocess.run(['sudo', '-n', 'true'], capture_output=True)
sudo_ok = sudo_check.returncode == 0
print("Passwordless sudo:", "AVAILABLE" if sudo_ok else "NOT AVAILABLE")

# Check key paths (handling permissions)
paths_to_check = {
    'suricata.yaml': CONFIG_FILE,
    'Log directory': LOG_DIR,
    'Rules directory': RULES_DIR,
}
print("\n--- System paths ---")
for name, path in paths_to_check.items():
    try:
        exists = path.exists()
    except PermissionError:
        result = subprocess.run(['sudo', 'test', '-e', str(path)], capture_output=True)
        exists = result.returncode == 0

    status = "OK" if exists else "NOT FOUND"
    print(f"{name:25s}: {path} [{status}]")

# Check network interface
iface_result = subprocess.run(['ip', 'route'], capture_output=True, text=True)
default_iface = None
for line in iface_result.stdout.splitlines():
    if 'default' in line:
        parts = line.split()
        if 'dev' in parts:
            default_iface = parts[parts.index('dev') + 1]
            break
print(f"\nMain network interface: {default_iface or 'Not detected'}")

---
## Section 2: Reviewing `/etc/suricata/suricata.yaml`

### Why does this configuration matter?

`suricata.yaml` is Suricata's central configuration file. It determines:
- **What traffic** to analyze (`HOME_NET`, `EXTERNAL_NET`)
- **Where to look** for detection rules (`default-rule-path`, `rule-files`)
- **How to log** detected events (`outputs`: `fast.log`, `eve.json`)
- **Which protocols** to decode and analyze

### Key parameters to understand:

| Parameter | Function | Impact on detection |
|-----------|---------|----------------------|
| `HOME_NET` | Defines the internal "protected" network | `$HOME_NET` rules only trigger when the destination/source belongs to this network |
| `EXTERNAL_NET` | Defines "external" traffic | `$EXTERNAL_NET` rules trigger for traffic outside HOME_NET |
| `default-rule-path` | Base rules directory | All relative paths in `rule-files` are relative to this directory |
| `rule-files` | List of rule files | Only the files listed here are loaded |
| `fast.log` | Plain-text alert log | Useful for quick lookups; format: `timestamp  [**] [sid:rev] signature [**] src->dst` |
| `eve.json` | Enriched JSON log | Contains all metadata: IPs, ports, protocols, payload, timestamps |

In [ ]:
# ── Read and display key sections of suricata.yaml ──────────────────────────
def read_yaml_section(filepath, keywords, context_lines=2):
    """Extracts lines containing the given keywords, with context."""
    content = read_file_safe(filepath)
    if not content:
        print(f"❌ Cannot read: {filepath}")
        return {}

    lines = content.splitlines()
    results = {}
    for kw in keywords:
        matches = []
        for i, line in enumerate(lines):
            if kw in line:
                start = max(0, i - context_lines)
                end   = min(len(lines), i + context_lines + 1)
                block = lines[start:end]
                matches.append((i + 1, block))
        results[kw] = matches
    return results

keywords = ['HOME_NET', 'EXTERNAL_NET', 'default-rule-path', 'rule-files',
            'fast.log', 'eve.json', 'community-id']

sections = read_yaml_section(CONFIG_FILE, keywords)

print("=" * 70)
print("CURRENT suricata.yaml CONFIGURATION (relevant sections)")
print("=" * 70)

for kw, matches in sections.items():
    if matches:
        print(f"\n▶ {kw}:")
        for lineno, block in matches[:2]:  # max 2 occurrences per keyword
            print(f"  [line {lineno}]")
            for l in block:
                print(f"    {l}")
    else:
        print(f"\n⚠️  '{kw}' not found in suricata.yaml")

In [ ]:
# ── Extract and explain HOME_NET / EXTERNAL_NET ───────────────────────────────
content = read_file_safe(CONFIG_FILE)
if content:
    home_lines = [l.strip() for l in content.splitlines() if 'HOME_NET:' in l]
    ext_lines  = [l.strip() for l in content.splitlines() if 'EXTERNAL_NET:' in l]
    rules_path = [l.strip() for l in content.splitlines() if 'default-rule-path' in l]
    rule_files = [l.strip() for l in content.splitlines() if 'suricata.rules' in l or
                  (l.strip().startswith('-') and '.rules' in l)]

    print("📋 NETWORK CONFIGURATION:")
    print(f"   HOME_NET     : {home_lines[0] if home_lines else 'Not found'}")
    print(f"   EXTERNAL_NET : {ext_lines[0] if ext_lines else 'Not found'}")
    print(f"   Rules path   : {rules_path[0] if rules_path else 'Not found'}")

    print("\n📋 RULE FILES:")
    for rf in rule_files[:5]:
        print(f"   {rf}")

    print("\n" + "─" * 60)
    print("💡 PEDAGOGICAL EXPLANATION:")
    print("""
  • HOME_NET defines which networks are "internal" or "protected".
    - Rules with $HOME_NET as destination detect attacks TOWARD your network.
    - If a PCAP has IPs outside the HOME_NET range, many rules
      targeting your network will NOT trigger → possible false negatives.

  • EXTERNAL_NET is usually 'any' or '!$HOME_NET' (any outside IP).
    - Rules with $EXTERNAL_NET as source detect traffic from outside.
    - If HOME_NET is very broad (e.g. 'any'), EXTERNAL_NET ends up empty
      → rules for external origin never trigger → false negatives.

  • Practical impact:
    - For the practice PCAPs, verify that the PCAP's IPs are
      covered by HOME_NET or EXTERNAL_NET as appropriate.
    - Setting HOME_NET to 'any' can generate more alerts but
      also more false positives.
    """)
else:
    print("⚠️  Could not read suricata.yaml")

In [ ]:
# ── Modify suricata.yaml to use suricata.rules ─────────────────────────
import re
import shutil as _shutil

TARGET_RULES_PATH = '/var/lib/suricata/rules'
TARGET_RULES_FILE = 'suricata.rules'

def modify_suricata_yaml(config_path: Path) -> bool:
    """
    Adjusts suricata.yaml so that:
      1. default-rule-path points to /var/lib/suricata/rules
      2. rule-files contains only suricata.rules
    Returns True if any change was made.
    """
    content = read_file_safe(config_path)
    if not content:
        print(f"❌ Cannot read {config_path}")
        return False

    original = content

    # Backup
    backup = config_path.with_suffix('.yaml.bak')
    result_backup = subprocess.run(['sudo', 'test', '-e', str(backup)], capture_output=True)
    if result_backup.returncode != 0:
        subprocess.run(['sudo', 'cp', str(config_path), str(backup)], check=True)
        print(f"💾 Backup saved to: {backup}")

    # 1. Adjust default-rule-path
    content = re.sub(
        r'(default-rule-path:\s*).*',
        f'\\g<1>{TARGET_RULES_PATH}',
        content
    )

    # 2. Insert/replace rule-files block
    rule_files_block = f"""rule-files:
  - {TARGET_RULES_FILE}"""

    if re.search(r'^rule-files:', content, re.MULTILINE):
        content = re.sub(
            r'^rule-files:.*?(?=^[a-z])',
            rule_files_block + '\n\n',
            content,
            flags=re.MULTILINE | re.DOTALL
        )
    else:
        content += f"\n{rule_files_block}\n"

    if content != original:
        # Write with sudo
        import tempfile, subprocess as _sp
        with tempfile.NamedTemporaryFile(mode='w', suffix='.yaml', delete=False) as tmp:
            tmp.write(content)
            tmp_path = tmp.name
        _sp.run(['sudo', 'cp', tmp_path, str(config_path)], check=True)
        _sp.run(['sudo', 'chmod', '640', str(config_path)], check=True)
        os.unlink(tmp_path)
        print("✅ suricata.yaml modified successfully")
        return True
    else:
        print("ℹ️  suricata.yaml was already configured correctly")
        return False

changed = modify_suricata_yaml(CONFIG_FILE)

# ── Validate changes ────────────────────────────────────────────────────────────
content_after = read_file_safe(CONFIG_FILE)
if content_after:
    rule_path_ok = TARGET_RULES_PATH in content_after
    rule_file_ok = TARGET_RULES_FILE in content_after
    print(f"\n🔍 Post-modification validation:")
    print(f"   {'✅' if rule_path_ok else '❌'} default-rule-path = {TARGET_RULES_PATH}")
    print(f"   {'✅' if rule_file_ok else '❌'} rule-files contains {TARGET_RULES_FILE}")

    # Show modified section
    lines = content_after.splitlines()
    for i, line in enumerate(lines):
        if 'default-rule-path' in line or 'rule-files' in line or TARGET_RULES_FILE in line:
            print(f"   [line {i+1}] {line}")
else:
    print("⚠️  Cannot validate changes - unable to read the file")

---
## Section 3: Rule Update with `suricata-update`

### Why update rules?

Suricata uses **signatures** (rules) to detect threats. The default installation rules are limited. `suricata-update` downloads and manages rule collections from public sources:

- **ET Open (Emerging Threats)**: The most comprehensive rules, community-maintained
- **PT Research**: Positive Technologies Research
- **OISF**: Open Information Security Foundation

### Impact of suricata-update:

| Without updating | With suricata-update (ET Open) |
|----------------|-------------------------------|
| ~100-500 basic rules | 40,000–50,000 rules |
| Limited detection | Broad threat coverage |
| No WannaCry/EternalBlue rules | Specific MS17-010, SMBv1 signatures |
| No rules for recent scans | Rules for Nmap, Masscan, etc. |

> **Note:** More rules = more detection capability, but also higher CPU/RAM usage.

In [ ]:
def count_rules(rules_file: Path) -> dict:
    """Counts active, commented, and total rules in a .rules file."""
    try:
        exists = rules_file.exists()
    except PermissionError:
        result = subprocess.run(['sudo', 'test', '-e', str(rules_file)], capture_output=True)
        exists = result.returncode == 0

    if not exists:
        return {'active': 0, 'commented': 0, 'total': 0}

    content = read_file_safe(rules_file)
    if not content:
        return {'active': 0, 'commented': 0, 'total': 0}

    lines = content.splitlines()
    active    = sum(1 for l in lines if l.strip().startswith('alert'))
    commented = sum(1 for l in lines if l.strip().startswith('#alert'))
    return {'active': active, 'commented': commented, 'total': active + commented}

rules_file = RULES_DIR / 'suricata.rules'
before = count_rules(rules_file)

try:
    file_exists = rules_file.exists()
except PermissionError:
    result = subprocess.run(['sudo', 'test', '-e', str(rules_file)], capture_output=True)
    file_exists = result.returncode == 0

print("=" * 70)
print("RULE STATE BEFORE UPDATING")
print("=" * 70)
print(f"\nActive rules:     {before['active']:>10,}")
print(f"Commented rules:  {before['commented']:>9,}")
print(f"Total:            {before['total']:>10,}")
print(f"File:             {rules_file}")
print(f"Exists:           {'Yes' if file_exists else 'No'}")

In [ ]:
# ── Run suricata-update ───────────────────────────────────────────────────
print(" Updating rule sources...")
print("\\n$ sudo suricata-update update-sources")
result_sources = subprocess.run(
    ['sudo', 'suricata-update', 'update-sources'],
    capture_output=True, text=True
)
if result_sources.returncode == 0:
    print("Rule sources updated")
else:
    print(f" update-sources: {result_sources.stderr[:200]}")

print("\\nEnabling ET Open...")
print("\\n$ sudo suricata-update enable-source et/open")
result_enable = subprocess.run(
    ['sudo', 'suricata-update', 'enable-source', 'et/open'],
    capture_output=True, text=True
)
if result_enable.returncode == 0:
    print(" ET Open enabled")
else:
    print(f" {result_enable.stdout[:200] or result_enable.stderr[:200]}")

print("\\nDownloading and compiling rules (may take 1-2 minutes)...")
print(f"\\n$ sudo suricata-update --no-merge --output {RULES_DIR}")
result_update = subprocess.run(
    ['sudo', 'suricata-update', '--no-merge', '--output', str(RULES_DIR)],
    capture_output=True, text=True
)

# Show output summary
output_lines = (result_update.stdout + result_update.stderr).splitlines()
for line in output_lines:
    if any(kw in line.lower() for kw in ['loaded', 'rules', 'enabled', 'disabled',
                                          'skipped', 'sources', 'fetching', 'ok', 'error']):
        print(f"  {line.strip()}")

if result_update.returncode == 0:
    print("\\n suricata-update completed")
else:
    print(f"\\n suricata-update finished with code {result_update.returncode}")

In [ ]:
after = count_rules(rules_file)

print("=" * 70)
print("RULE COMPARISON")
print("=" * 70)
print(f"\n{'Metric':<22} {'Before':>10} {'After':>10} {'Delta':>10}")
print("-" * 55)
for key in ['active', 'commented', 'total']:
    delta = after[key] - before[key]
    delta_str = f"+{delta:,}" if delta >= 0 else f"{delta:,}"
    print(f"  {key.capitalize():<20} {before[key]:>10,} {after[key]:>10,} {delta_str:>10}")

print()
if after['active'] > before['active']:
    print(f"Added {after['active'] - before['active']:,} new active rules")
elif after['active'] == 0:
    print("WARNING: No rules found. Check connectivity.")
else:
    print(f"Number of active rules: {after['active']:,}")

# Validate configuration with suricata -T
print("\nValidating Suricata configuration...")
test_result = subprocess.run(
    ['sudo', 'suricata', '-T', '-c', str(CONFIG_FILE), '-v'],
    capture_output=True, text=True
)

if test_result.returncode == 0:
    for line in (test_result.stdout + test_result.stderr).splitlines():
        if any(kw in line for kw in ['Loading', 'rules', 'Configuration', 'Done', 'Error', 'OK']):
            print(f"  {line.strip()}")
    print("\nValid configuration: suricata.yaml is syntactically correct")
else:
    print(f"CONFIGURATION ERROR:\n{test_result.stderr[:500]}")

---
## Section 4: Offline Analysis — PCAP 1: Webserver Scans and Probes

### Case Description

This PCAP contains traffic captured from **scans and probes against web servers**.
Typical traffic includes:
- Port scans (TCP SYN scan, Nmap, Masscan)
- Reconnaissance HTTP requests (dirbusting, nikto)
- Attempts to exploit known web vulnerabilities
- Service version scans

### What to expect from Suricata?

| Expected alert type | Typical signature | Protocol |
|------------------------|--------------|-----------|
| Port scan | ET SCAN Nmap... / ET SCAN Masscan | TCP |
| HTTP reconnaissance | ET POLICY HTTP HEAD Request | HTTP |
| Web exploits | ET WEB_SPECIFIC_APPS ... | HTTP |
| Service reconnaissance | ET SCAN ... version scan | TCP |

In [ ]:
# URLs for PCAP download (verified and working on Google Colab)
PCAP1_URL      = 'https://www.malware-traffic-analysis.net/2024/11/24/2024-11-24-webserver-scans-and-probes.pcap.zip'
PCAP1_ZIP      = PCAP_DIR / '2024-11-24-webserver-scans-and-probes.pcap.zip'
PCAP1_PASSWORD = 'infected_20241124'
PCAP1_NAME     = '2024-11-24-webserver-scans-and-probes.pcap'

PCAP2_URL      = 'https://www.malware-traffic-analysis.net/2017/05/18/2017-05-18-WannaCry-ransomware-using-EnternalBlue-exploit.pcap.zip'
PCAP2_ZIP      = PCAP_DIR / '2017-05-18-WannaCry-ransomware-using-EnternalBlue-exploit.pcap.zip'
PCAP2_PASSWORD = 'infected_20170518'
PCAP2_NAME     = '2017-05-18-WannaCry-ransomware-using-EnternalBlue-exploit.pcap'

def download_pcap(url: str, dest_zip: Path, pcap_password: str,
                  expected_name: str = None) -> Path:
    """Downloads, extracts, and validates a PCAP. Returns the path to the .pcap."""

    print(f"Processing: {expected_name or url}")
    print("-" * 70)

    # Check whether the extracted PCAP already exists
    existing = list(PCAP_DIR.glob('*.pcap'))
    for p in existing:
        if expected_name and expected_name in p.name:
            print(f"PCAP found: {p.name} ({p.stat().st_size / (1024*1024):.1f} MB)")
            return p

    # Download ZIP
    if not dest_zip.exists():
        print(f"Downloading: {url}")
        dl_result = subprocess.run(
            ['wget', '-q', '--show-progress', '-O', str(dest_zip), url],
            capture_output=True, text=True, timeout=600
        )
        if dl_result.returncode != 0:
            print(f"ERROR: Download failed (code {dl_result.returncode})")
            print(f"Try downloading manually: {url}")
            print(f"Password: {pcap_password}")
            return None
        print(f"ZIP downloaded: {dest_zip.name} ({dest_zip.stat().st_size / (1024*1024):.1f} MB)")
    else:
        print(f"ZIP already exists: {dest_zip.name}")

    # Extract with password
    print(f"Extracting with password...")
    unzip_result = subprocess.run(
        ['unzip', '-P', pcap_password, '-o', str(dest_zip), '-d', str(PCAP_DIR)],
        capture_output=True, text=True, timeout=120
    )

    if unzip_result.returncode != 0:
        print(f"ERROR extracting: {unzip_result.stderr[:200]}")
        return None

    # Find the resulting PCAP
    pcaps = list(PCAP_DIR.glob('*.pcap'))
    if pcaps:
        if expected_name:
            for p in pcaps:
                if expected_name in p.name:
                    print(f"OK: {p.name} ({p.stat().st_size / (1024*1024):.1f} MB)")
                    return p

        pcap = max(pcaps, key=lambda p: p.stat().st_mtime)
        print(f"OK: {pcap.name} ({pcap.stat().st_size / (1024*1024):.1f} MB)")
        return pcap
    else:
        print(f"ERROR: No PCAP found in {PCAP_DIR}")
        return None

# Download PCAP 1
print("\n" + "=" * 70)
print("DOWNLOADING PCAP 1: Webserver Scans and Probes")
print("=" * 70)
pcap1_path = download_pcap(PCAP1_URL, PCAP1_ZIP, PCAP1_PASSWORD, PCAP1_NAME)
PCAP1_PATH = pcap1_path

In [ ]:
PCAP1_RESULTS = RESULTS_DIR / 'pcap1_webserver'
PCAP1_RESULTS.mkdir(parents=True, exist_ok=True)

def run_suricata_offline(pcap_path: Path, output_dir: Path,
                          config: Path = CONFIG_FILE) -> bool:
    """Runs Suricata in offline mode on a given PCAP."""
    # Safe file existence check
    try:
        pcap_exists = pcap_path.exists() if pcap_path else False
    except PermissionError:
        result = subprocess.run(['sudo', 'test', '-e', str(pcap_path)], capture_output=True)
        pcap_exists = result.returncode == 0

    if pcap_path is None or not pcap_exists:
        print(f"ERROR: PCAP not available: {pcap_path}")
        return False

    # Clean up previous results
    for f in output_dir.glob('*.log'):
        f.unlink()
    for f in output_dir.glob('*.json'):
        f.unlink()

    print(f"Analyzing: {pcap_path.name}")
    print(f"Results in: {output_dir}")
    print("Processing PCAP (may take a while)...")

    result = subprocess.run(
        ['sudo', 'suricata',
         '-r', str(pcap_path),
         '-c', str(config),
         '-l', str(output_dir),
         '-k', 'none',
         '--set', 'outputs.1.fast.enabled=yes'],
        capture_output=True, text=True, timeout=600
    )

    if result.returncode == 0 or output_dir.joinpath('fast.log').exists():
        fast_log = output_dir / 'fast.log'
        eve_json = output_dir / 'eve.json'

        # Safe file reading
        alerts = 0
        if fast_log.exists():
            fast_content = read_file_safe(fast_log)
            alerts = fast_content.count('[**]') if fast_content else 0

        events = 0
        if eve_json.exists():
            try:
                with open(eve_json, errors='replace') as f:
                    events = sum(1 for _ in f)
            except PermissionError:
                result_eve = subprocess.run(['sudo', 'wc', '-l', str(eve_json)],
                                           capture_output=True, text=True)
                if result_eve.returncode == 0:
                    events = int(result_eve.stdout.split()[0])

        print(f"\nAnalysis complete:")
        print(f"  fast.log: {'OK' if fast_log.exists() else 'NO'}")
        print(f"  eve.json: {'OK' if eve_json.exists() else 'NO'}")
        print(f"  Alerts: {alerts}")
        print(f"  Events: {events}")
        return True
    else:
        print(f"ERROR: Suricata failed (code {result.returncode})")
        return False

if PCAP1_PATH:
    success_pcap1 = run_suricata_offline(PCAP1_PATH, PCAP1_RESULTS)
else:
    print("ERROR: PCAP 1 not available for analysis")
    success_pcap1 = False

In [ ]:
import re

def parse_fast_log(fast_log_path: Path, max_alerts: int = None) -> list:
    """Parses fast.log and returns a list of dicts with the alerts."""
    alerts = []

    # Safe file existence check
    try:
        exists = fast_log_path.exists()
    except PermissionError:
        result = subprocess.run(['sudo', 'test', '-e', str(fast_log_path)], capture_output=True)
        exists = result.returncode == 0

    if not exists:
        return alerts

    # Safe file reading
    content = read_file_safe(fast_log_path)
    if not content:
        return alerts

    pattern = re.compile(
        r'(\d{2}/\d{2}/\d{4}-[\d:]+\.\d+)\s+'   # timestamp
        r'\[\*\*\]\s+\[(.+?)\]\s+'              # [gid:sid:rev]
        r'(.+?)\s+\[\*\*\]\s+'                   # signature
        r'\[Classification:\s*(.+?)\]\s+'          # classification
        r'\[Priority:\s*(\d+)\]\s+'               # priority
        r'\{(\w+)\}\s+'                            # protocol
        r'([\d.]+:\d+)\s*->\s*([\d.]+:\d+)'     # src -> dst
    )

    for line in content.splitlines():
        m = pattern.match(line.strip())
        if m:
            alerts.append({
                'timestamp':      m.group(1),
                'sid':            m.group(2),
                'signature':      m.group(3),
                'classification': m.group(4),
                'priority':       int(m.group(5)),
                'protocol':       m.group(6),
                'src':            m.group(7),
                'dst':            m.group(8),
                'raw':            line.strip()
            })
        elif '[**]' in line:
            alerts.append({'raw': line.strip(), 'signature': line.strip(), 'priority': 99})

    if max_alerts is None:
        return alerts
    return alerts[:max_alerts]

fast1 = PCAP1_RESULTS / 'fast.log'
alerts1 = parse_fast_log(fast1)

print("=" * 70)
print("FAST.LOG ALERTS - PCAP 1: WEBSERVER SCANS")
print("=" * 70)
print(f"\nTotal alerts: {len(alerts1)}\n")

# Top signatures
from collections import Counter
sig_counter1 = Counter()

for a in alerts1:
    sig = a.get('signature', 'N/A')[:60]
    sig_counter1[sig] += 1

print("Top detected signatures:")
print("-" * 70)
for sig, cnt in sig_counter1.most_common(15):
    print(f"{cnt:>5} occurrences | {sig}")

# Show ALL alerts
print(f"\nAlert details ({len(alerts1)} total):")
print("-" * 70)
for i, a in enumerate(alerts1, 1):
    raw = a.get('raw', 'N/A')
    if len(raw) > 150:
        raw = raw[:147] + "..."
    print(f"{i:>4}. {raw}")

# Priority summary
prio_counter = Counter(a.get('priority', 99) for a in alerts1)
print("\nDistribution by priority:")
print("-" * 70)
for p in sorted(prio_counter.keys()):
    label = {1: 'CRITICAL', 2: 'HIGH', 3: 'MEDIUM'}.get(p, f'LEVEL {p}')
    print(f"Priority {p} ({label:10s}): {prio_counter[p]:>6} alerts")

# Summary by protocol
proto_counter = Counter(a.get('protocol', 'N/A') for a in alerts1 if 'protocol' in a)
print("\nAlerts by protocol:")
print("-" * 70)
for proto, cnt in proto_counter.most_common():
    print(f"{proto:>10s}: {cnt:>6} alerts")

In [ ]:
import pandas as pd

def load_eve_json(eve_path: Path, max_lines: int = None) -> pd.DataFrame:
    """Loads eve.json into a DataFrame, expanding the 'alert' field."""
    records = []

    # Safe file existence check
    try:
        exists = eve_path.exists()
    except PermissionError:
        result = subprocess.run(['sudo', 'test', '-e', str(eve_path)], capture_output=True)
        exists = result.returncode == 0

    if not exists:
        return pd.DataFrame()

    # Try to open file safely
    try:
        with open(eve_path, errors='replace') as f:
            for i, line in enumerate(f):
                if max_lines and i >= max_lines:
                    break
                try:
                    records.append(json.loads(line))
                except json.JSONDecodeError:
                    pass
    except PermissionError:
        # Fall back to sudo
        result = subprocess.run(['sudo', 'cat', str(eve_path)],
                              capture_output=True, text=True)
        if result.returncode == 0:
            for i, line in enumerate(result.stdout.splitlines()):
                if max_lines and i >= max_lines:
                    break
                try:
                    records.append(json.loads(line))
                except json.JSONDecodeError:
                    pass

    if not records:
        return pd.DataFrame()

    df = pd.json_normalize(records, sep='_')
    return df

eve1_path = PCAP1_RESULTS / 'eve.json'
df1 = load_eve_json(eve1_path)

if not df1.empty:
    print("=" * 70)
    print("EVE.JSON EVENTS - PCAP 1: WEBSERVER SCANS")
    print("=" * 70)
    print(f"\nTotal events: {len(df1)}\n")

    # Event types
    if 'event_type' in df1.columns:
        print("Distribution by event type:")
        print("-" * 70)
        for etype, cnt in df1['event_type'].value_counts().items():
            print(f"{etype:>20s}: {cnt:>8} events")

    # Show ALL alerts in an organized way
    if 'event_type' in df1.columns:
        df_alerts1 = df1[df1['event_type'] == 'alert'].copy()

        if not df_alerts1.empty:
            print(f"\nALERT DETAILS ({len(df_alerts1)} total):")
            print("-" * 70)

            # Select relevant columns
            display_cols = ['timestamp', 'src_ip', 'src_port', 'dest_ip', 'dest_port', 'proto']
            alert_cols = ['alert_signature', 'alert_category', 'alert_severity']

            available_display = [c for c in display_cols if c in df_alerts1.columns]
            available_alert = [c for c in alert_cols if c in df_alerts1.columns]

            # Show all alerts, numbered
            for i, (idx, row) in enumerate(df_alerts1.iterrows(), 1):
                timestamp = row.get('timestamp', 'N/A')
                src_ip = row.get('src_ip', 'N/A')
                src_port = row.get('src_port', 'N/A')
                dest_ip = row.get('dest_ip', 'N/A')
                dest_port = row.get('dest_port', 'N/A')
                proto = row.get('proto', 'N/A')
                signature = row.get('alert_signature', 'N/A')
                category = row.get('alert_category', 'N/A')
                severity = row.get('alert_severity', 'N/A')

                print(f"{i:>4}. [{timestamp}] {src_ip}:{src_port} -> {dest_ip}:{dest_port} ({proto})")
                print(f"      Signature: {signature}")
                print(f"      Category: {category} | Severity: {severity}\n")

            # Protocol summary
            print("Alerts by protocol:")
            print("-" * 70)
            if 'proto' in df_alerts1.columns:
                for proto, cnt in df_alerts1['proto'].value_counts().items():
                    print(f"{proto:>20s}: {cnt:>8} alerts")

            # Top source IPs
            if 'src_ip' in df_alerts1.columns:
                print("\nTop 10 alert source IPs:")
                print("-" * 70)
                for rank, (ip, cnt) in enumerate(df_alerts1['src_ip'].value_counts().head(10).items(), 1):
                    print(f"{rank:>2}. {ip:>20s}: {cnt:>8} alerts")

            # Top signatures
            if 'alert_signature' in df_alerts1.columns:
                print("\nTop 10 most frequent signatures:")
                print("-" * 70)
                for rank, (sig, cnt) in enumerate(df_alerts1['alert_signature'].value_counts().head(10).items(), 1):
                    sig_short = sig[:60] if len(str(sig)) > 60 else sig
                    print(f"{rank:>2}. {sig_short:60s}: {cnt:>8} alerts")
        else:
            print("\nNo alerts found")
    else:
        print("\n'event_type' column not found")
else:
    print("Could not load events from eve.json")

In [ ]:
# ── PCAP 1 results visualization ────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib
import warnings
warnings.filterwarnings('ignore')

# Use inline backend for static output — compatible with all Jupyter environments (no ipympl required)
%matplotlib inline
plt.rcParams.update({'figure.figsize': (14, 5), 'figure.dpi': 100,
                     'font.size': 10})

if not df1.empty and 'event_type' in df1.columns:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle('PCAP 1: Webserver Scans and Probes — Suricata Analysis', fontsize=13)

    # ── Panel 1: Event types ─────────────────────────────────────────────
    event_counts = df1['event_type'].value_counts().head(8)
    axes[0].barh(event_counts.index, event_counts.values, color='steelblue')
    axes[0].set_title('Event Types')
    axes[0].set_xlabel('Count')
    axes[0].invert_yaxis()

    # ── Panel 2: Top signatures ───────────────────────────────────────────────────
    df_al1 = df1[df1['event_type'] == 'alert'] if 'event_type' in df1.columns else pd.DataFrame()
    if not df_al1.empty and 'alert_signature' in df_al1.columns:
        top_sigs = df_al1['alert_signature'].value_counts().head(8)
        labels = [s[:35] + '...' if len(str(s)) > 35 else str(s) for s in top_sigs.index]
        axes[1].barh(labels, top_sigs.values, color='coral')
        axes[1].set_title('Top Alert Signatures')
        axes[1].set_xlabel('Count')
        axes[1].invert_yaxis()
    else:
        axes[1].text(0.5, 0.5, 'No alert\ndata', ha='center', va='center',
                     transform=axes[1].transAxes)
        axes[1].set_title('Top Alert Signatures')

    # ── Panel 3: Protocols ───────────────────────────────────────────────────
    if 'proto' in df1.columns:
        proto_counts = df1['proto'].value_counts().head(6)
        axes[2].pie(proto_counts.values,
                    labels=[str(p) for p in proto_counts.index],
                    autopct='%1.1f%%', startangle=90,
                    colors=plt.cm.Set3.colors[:len(proto_counts)])
        axes[2].set_title('Protocol Distribution')
    else:
        axes[2].text(0.5, 0.5, 'No protocol\ndata', ha='center', va='center',
                     transform=axes[2].transAxes)

    plt.tight_layout()
    plt.savefig(str(RESULTS_DIR / 'pcap1_analysis.png'), bbox_inches='tight')
    plt.show()
    print("📊 Chart saved to:", RESULTS_DIR / 'pcap1_analysis.png')
else:
    print("⚠️  Not enough data to visualize. Check the PCAP analysis.")

---
## Section 5: Offline Analysis — PCAP 2: WannaCry / EternalBlue

### Case Description

This historical PCAP (May 2017) contains traffic captured during the **WannaCry ransomware** attack, which exploited the **EternalBlue (MS17-010)** vulnerability.

### Technical Context

| Aspect | Detail |
|---------|---------|
| Vulnerability | MS17-010 (SMBv1, port 445) |
| Exploit | EternalBlue (leaked from the NSA) |
| Payload | WannaCry ransomware |
| Propagation | Self-propagation across Windows networks |
| CVE | CVE-2017-0144, CVE-2017-0145 |

### What to expect from Suricata?

| Expected alert type | Typical ET Open signature | Protocol |
|------------------------|---------------------|-----------|
| SMBv1 exploitation | `ET EXPLOIT MS17-010 EternalBlue` | SMB/TCP |
| SMB scanning | `ET SCAN SMB Brute Force` | TCP/445 |
| Worm propagation | `ET WORM` | TCP |
| C&C traffic | `ET MALWARE` | TCP |
| DoublePulsar activity | `ET EXPLOIT DoublePulsar` | SMB |

> **Teaching note:** The key difference from PCAP 1 is the **dominant protocol** (SMB vs HTTP) and the **nature of the alerts** (active exploitation vs reconnaissance).

In [ ]:
print("\n" + "=" * 70)
print("DOWNLOADING PCAP 2: WannaCry / EternalBlue")
print("=" * 70)

PCAP2_PATH = None

# Check whether a WannaCry PCAP already exists
for existing in PCAP_DIR.glob('*.pcap'):
    if 'WannaCry' in existing.name or 'EternalBlue' in existing.name or '2017' in existing.name:
        print(f"PCAP found: {existing.name} ({existing.stat().st_size / (1024*1024):.1f} MB)")
        PCAP2_PATH = existing
        break

# If it doesn't exist, download it
if PCAP2_PATH is None:
    pcap2_path = download_pcap(PCAP2_URL, PCAP2_ZIP, PCAP2_PASSWORD, PCAP2_NAME)
    PCAP2_PATH = pcap2_path

if PCAP2_PATH:
    print(f"\nPCAP 2 available: {PCAP2_PATH.name}")
else:
    print("\nERROR: PCAP 2 not available")
    print(f"URL: {PCAP2_URL}")
    print(f"Password: {PCAP2_PASSWORD}")
    print("Try downloading manually and place it in:", PCAP_DIR)

In [ ]:
PCAP2_RESULTS = RESULTS_DIR / 'pcap2_wannacry'
PCAP2_RESULTS.mkdir(parents=True, exist_ok=True)

print("\n" + "=" * 70)
print("ANALYZING PCAP 2: WANNACRY / ETERNALBLUE")
print("=" * 70)

if PCAP2_PATH:
    success_pcap2 = run_suricata_offline(PCAP2_PATH, PCAP2_RESULTS)
else:
    print("ERROR: PCAP 2 not available for analysis")
    success_pcap2 = False

In [ ]:
fast2 = PCAP2_RESULTS / 'fast.log'
alerts2 = parse_fast_log(fast2)

print("=" * 70)
print("FAST.LOG ALERTS - PCAP 2: WANNACRY / ETERNALBLUE")
print("=" * 70)
print(f"\nTotal alerts: {len(alerts2)}\n")

sig_counter2 = Counter()
for a in alerts2:
    sig = a.get('signature', 'N/A')[:60]
    sig_counter2[sig] += 1

print("Top detected signatures:")
print("-" * 70)
for sig, cnt in sig_counter2.most_common(15):
    print(f"{cnt:>5} occurrences | {sig}")

# Show ALL alerts
print(f"\nAlert details ({len(alerts2)} total):")
print("-" * 70)
for i, a in enumerate(alerts2, 1):
    raw = a.get('raw', 'N/A')
    if len(raw) > 150:
        raw = raw[:147] + "..."
    print(f"{i:>4}. {raw}")

prio_counter2 = Counter(a.get('priority', 99) for a in alerts2)
print("\nDistribution by priority:")
print("-" * 70)
for p in sorted(prio_counter2.keys()):
    label = {1: 'CRITICAL', 2: 'HIGH', 3: 'MEDIUM'}.get(p, f'LEVEL {p}')
    print(f"Priority {p} ({label:10s}): {prio_counter2[p]:>6} alerts")

# Summary by protocol
proto_counter2 = Counter(a.get('protocol', 'N/A') for a in alerts2 if 'protocol' in a)
print("\nAlerts by protocol:")
print("-" * 70)
for proto, cnt in proto_counter2.most_common():
    print(f"{proto:>10s}: {cnt:>6} alerts")

In [ ]:
eve2_path = PCAP2_RESULTS / 'eve.json'
df2 = load_eve_json(eve2_path)

if not df2.empty:
    print("=" * 70)
    print("EVE.JSON EVENTS - PCAP 2: WANNACRY / ETERNALBLUE")
    print("=" * 70)
    print(f"\nTotal events: {len(df2)}\n")

    # Event types
    if 'event_type' in df2.columns:
        print("Distribution by event type:")
        print("-" * 70)
        for etype, cnt in df2['event_type'].value_counts().items():
            print(f"{etype:>20s}: {cnt:>8} events")

    # Show ALL alerts in an organized way
    if 'event_type' in df2.columns:
        df_alerts2 = df2[df2['event_type'] == 'alert'].copy()

        if not df_alerts2.empty:
            print(f"\nALERT DETAILS ({len(df_alerts2)} total):")
            print("-" * 70)

            # Show all alerts, numbered
            for i, (idx, row) in enumerate(df_alerts2.iterrows(), 1):
                timestamp = row.get('timestamp', 'N/A')
                src_ip = row.get('src_ip', 'N/A')
                src_port = row.get('src_port', 'N/A')
                dest_ip = row.get('dest_ip', 'N/A')
                dest_port = row.get('dest_port', 'N/A')
                proto = row.get('proto', 'N/A')
                signature = row.get('alert_signature', 'N/A')
                category = row.get('alert_category', 'N/A')
                severity = row.get('alert_severity', 'N/A')

                print(f"{i:>4}. [{timestamp}] {src_ip}:{src_port} -> {dest_ip}:{dest_port} ({proto})")
                print(f"      Signature: {signature}")
                print(f"      Category: {category} | Severity: {severity}\n")

            # Protocol summary
            print("Alerts by protocol:")
            print("-" * 70)
            if 'proto' in df_alerts2.columns:
                for proto, cnt in df_alerts2['proto'].value_counts().items():
                    print(f"{proto:>20s}: {cnt:>8} alerts")

            # Top source IPs
            if 'src_ip' in df_alerts2.columns:
                print("\nTop 10 alert source IPs:")
                print("-" * 70)
                for rank, (ip, cnt) in enumerate(df_alerts2['src_ip'].value_counts().head(10).items(), 1):
                    print(f"{rank:>2}. {ip:>20s}: {cnt:>8} alerts")

            # Top signatures
            if 'alert_signature' in df_alerts2.columns:
                print("\nTop 10 most frequent signatures:")
                print("-" * 70)
                for rank, (sig, cnt) in enumerate(df_alerts2['alert_signature'].value_counts().head(10).items(), 1):
                    sig_short = sig[:60] if len(str(sig)) > 60 else sig
                    print(f"{rank:>2}. {sig_short:60s}: {cnt:>8} alerts")

            # Look for WannaCry-specific signatures
            print("\nSignatures related to WannaCry/EternalBlue:")
            print("-" * 70)
            wannacry_keywords = ['MS17-010', 'EternalBlue', 'DoublePulsar', 'SMB', 'EXPLOIT']
            for kw in wannacry_keywords:
                mask = df_alerts2['alert_signature'].str.contains(kw, case=False, na=False)
                count = mask.sum()
                if count > 0:
                    print(f"'{kw}': {count} alerts found")
        else:
            print("\nNo alerts found")
    else:
        print("\n'event_type' column not found")
else:
    print("Could not load events from eve.json")

In [ ]:
# ── PCAP 2 results visualization ────────────────────────────────────────
if not df2.empty and 'event_type' in df2.columns:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle('PCAP 2: WannaCry/EternalBlue — Suricata Analysis', fontsize=13)

    event_counts2 = df2['event_type'].value_counts().head(8)
    axes[0].barh(event_counts2.index, event_counts2.values, color='darkred')
    axes[0].set_title('Event Types')
    axes[0].set_xlabel('Count')
    axes[0].invert_yaxis()

    df_al2 = df2[df2['event_type'] == 'alert'] if 'event_type' in df2.columns else pd.DataFrame()
    if not df_al2.empty and 'alert_signature' in df_al2.columns:
        top_sigs2 = df_al2['alert_signature'].value_counts().head(8)
        labels2 = [s[:35] + '...' if len(str(s)) > 35 else str(s) for s in top_sigs2.index]
        axes[1].barh(labels2, top_sigs2.values, color='darkorange')
        axes[1].set_title('Top Alert Signatures')
        axes[1].set_xlabel('Count')
        axes[1].invert_yaxis()
    else:
        axes[1].text(0.5, 0.5, 'No alert\ndata', ha='center', va='center',
                     transform=axes[1].transAxes)
        axes[1].set_title('Top Alert Signatures')

    if 'proto' in df2.columns:
        proto_counts2 = df2['proto'].value_counts().head(6)
        axes[2].pie(proto_counts2.values,
                    labels=[str(p) for p in proto_counts2.index],
                    autopct='%1.1f%%', startangle=90,
                    colors=plt.cm.Set2.colors[:len(proto_counts2)])
        axes[2].set_title('Protocol Distribution')
    else:
        axes[2].text(0.5, 0.5, 'No protocol\ndata', ha='center', va='center',
                     transform=axes[2].transAxes)

    plt.tight_layout()
    plt.savefig(str(RESULTS_DIR / 'pcap2_analysis.png'), bbox_inches='tight')
    plt.show()
    print("📊 Chart saved to:", RESULTS_DIR / 'pcap2_analysis.png')
else:
    print("⚠️  Not enough data to visualize PCAP 2.")

---
## Section 6: Comparative Analysis — Webserver Scans vs WannaCry/EternalBlue

### Why compare these two PCAPs?

These two cases represent **opposite threat profiles**:

| Dimension | PCAP 1: Webserver Scans | PCAP 2: WannaCry |
|-----------|------------------------|-----------------|
| **Attack phase** | Reconnaissance | Active exploitation |
| **Dominant protocol** | HTTP/TCP | SMB (TCP/445) |
| **Target** | Public web servers | Windows systems on internal network |
| **MITRE technique** | T1046 (Network Service Scanning) | T1210 (Exploitation of Remote Services) |
| **Severity** | Medium–High | Critical |
| **Propagation** | Does not propagate | Self-propagating (worm) |

This comparison will let you answer the assignment's question:
> *What differences do you observe between the "activity profile" detected by Suricata in the webserver scans and probes PCAP versus the historical WannaCry/EternalBlue PCAP?*

In [ ]:
# ── Comparison table of the two PCAPs ────────────────────────────────────────
def get_pcap_stats(df: pd.DataFrame, pcap_name: str) -> dict:
    """Extracts summary statistics from an eve.json DataFrame."""
    if df.empty:
        return {'name': pcap_name, 'total_events': 0, 'alerts': 0,
                'top_protocol': 'N/A', 'top_signature': 'N/A',
                'unique_src_ips': 0, 'unique_dst_ips': 0,
                'categories': []}

    stats = {'name': pcap_name}
    stats['total_events'] = len(df)

    if 'event_type' in df.columns:
        df_al = df[df['event_type'] == 'alert']
        stats['alerts'] = len(df_al)

        if 'proto' in df.columns:
            stats['top_protocol'] = df['proto'].value_counts().index[0] \
                if not df['proto'].empty else 'N/A'
        else:
            stats['top_protocol'] = 'N/A'

        if 'alert_signature' in df_al.columns and not df_al.empty:
            stats['top_signature'] = df_al['alert_signature'].value_counts().index[0] \
                if not df_al['alert_signature'].empty else 'N/A'
        else:
            stats['top_signature'] = 'N/A'

        stats['unique_src_ips'] = df['src_ip'].nunique() if 'src_ip' in df.columns else 0
        stats['unique_dst_ips'] = df['dest_ip'].nunique() if 'dest_ip' in df.columns else 0

        if 'alert_category' in df_al.columns and not df_al.empty:
            stats['categories'] = df_al['alert_category'].value_counts().head(3).index.tolist()
        else:
            stats['categories'] = []
    else:
        stats.update({'alerts': 0, 'top_protocol': 'N/A', 'top_signature': 'N/A',
                      'unique_src_ips': 0, 'unique_dst_ips': 0, 'categories': []})

    return stats

stats1 = get_pcap_stats(df1, 'PCAP 1: Webserver Scans')
stats2 = get_pcap_stats(df2, 'PCAP 2: WannaCry/EternalBlue')

print("=" * 75)
print("COMPARISON TABLE — Webserver Scans vs WannaCry/EternalBlue")
print("=" * 75)
print(f"{'Metric':<35} {'PCAP 1 (Scans)':<20} {'PCAP 2 (WannaCry)':<20}")
print("-" * 75)

comparisons = [
    ('Total events (eve.json)',  'total_events'),
    ('Total alerts',             'alerts'),
    ('Dominant protocol',        'top_protocol'),
    ('Unique source IPs',        'unique_src_ips'),
    ('Unique destination IPs',   'unique_dst_ips'),
]
for label, key in comparisons:
    v1 = str(stats1.get(key, 'N/A'))
    v2 = str(stats2.get(key, 'N/A'))
    print(f"  {label:<33} {v1:<20} {v2:<20}")

print(f"  {'Most frequent signature':<33} {str(stats1['top_signature'])[:20]:<20} {str(stats2['top_signature'])[:20]:<20}")
print("=" * 75)

# Alert categories
print("\n📂 Detected alert categories:")
print(f"  PCAP 1: {', '.join(stats1['categories'][:3]) or 'N/A'}")
print(f"  PCAP 2: {', '.join(stats2['categories'][:3]) or 'N/A'}")

In [ ]:
# ── Comparative visualization ─────────────────────────────────────────────────
has_data1 = not df1.empty
has_data2 = not df2.empty

if has_data1 or has_data2:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle('Comparison: Webserver Scans vs WannaCry/EternalBlue', fontsize=13)

    # ── Panel 1: Event types side-by-side ───────────────────────────────
    all_types = set()
    if has_data1 and 'event_type' in df1.columns:
        all_types.update(df1['event_type'].value_counts().head(6).index.tolist())
    if has_data2 and 'event_type' in df2.columns:
        all_types.update(df2['event_type'].value_counts().head(6).index.tolist())

    all_types = sorted(all_types)
    x = range(len(all_types))
    w = 0.35

    counts1 = [df1['event_type'].value_counts().get(t, 0) if has_data1 and 'event_type' in df1.columns else 0
               for t in all_types]
    counts2 = [df2['event_type'].value_counts().get(t, 0) if has_data2 and 'event_type' in df2.columns else 0
               for t in all_types]

    bars1 = axes[0].bar([i - w/2 for i in x], counts1, w, label='PCAP 1 (Scans)', color='steelblue')
    bars2 = axes[0].bar([i + w/2 for i in x], counts2, w, label='PCAP 2 (WannaCry)', color='darkred')
    axes[0].set_xticks(list(x))
    axes[0].set_xticklabels(all_types, rotation=30, ha='right')
    axes[0].set_title('Event Types by PCAP')
    axes[0].set_ylabel('Count')
    axes[0].legend()

    # ── Panel 2: Top unique signatures per PCAP ──────────────────────────────
    top_pcap1_sigs = {}
    top_pcap2_sigs = {}

    if has_data1 and 'event_type' in df1.columns:
        df_al1 = df1[df1['event_type'] == 'alert']
        if 'alert_signature' in df_al1.columns and not df_al1.empty:
            top_pcap1_sigs = df_al1['alert_signature'].value_counts().head(5).to_dict()

    if has_data2 and 'event_type' in df2.columns:
        df_al2 = df2[df2['event_type'] == 'alert']
        if 'alert_signature' in df_al2.columns and not df_al2.empty:
            top_pcap2_sigs = df_al2['alert_signature'].value_counts().head(5).to_dict()

    if top_pcap1_sigs or top_pcap2_sigs:
        categories_labels = ['PCAP 1\n(Scans)', 'PCAP 2\n(WannaCry)']
        total_alerts = [stats1['alerts'], stats2['alerts']]
        colors_bar = ['steelblue', 'darkred']
        axes[1].bar(categories_labels, total_alerts, color=colors_bar, width=0.4)
        axes[1].set_title('Total Alerts by PCAP')
        axes[1].set_ylabel('Number of alerts')
        for i, v in enumerate(total_alerts):
            axes[1].text(i, v + max(total_alerts) * 0.02, str(v),
                         ha='center', fontweight='bold')
    else:
        axes[1].text(0.5, 0.5, 'No data\nto compare', ha='center', va='center',
                     transform=axes[1].transAxes)

    plt.tight_layout()
    plt.savefig(str(RESULTS_DIR / 'comparison.png'), bbox_inches='tight')
    plt.show()
    print("📊 Comparison chart saved to:", RESULTS_DIR / 'comparison.png')
else:
    print("⚠️  No data from either PCAP available to compare visually.")

In [ ]:
# ── HOME_NET/EXTERNAL_NET analysis and detection impact ─────────────────
print("=" * 70)
print("CONFIGURATION ANALYSIS — HOME_NET / EXTERNAL_NET")
print("=" * 70)

# Extract current values
home_net_value   = 'Not configured'
external_net_val = 'Not configured'

content = read_file_safe(CONFIG_FILE)
if content:
    for line in content.splitlines():
        if 'HOME_NET:' in line and not line.strip().startswith('#'):
            home_net_value = line.split(':', 1)[1].strip().strip('"')
        if 'EXTERNAL_NET:' in line and not line.strip().startswith('#'):
            external_net_val = line.split(':', 1)[1].strip().strip('"')

print(f"\n📋 Current configuration:")
print(f"   HOME_NET     = {home_net_value}")
print(f"   EXTERNAL_NET = {external_net_val}")

# Detect IPs in the PCAPs
def get_unique_ips(df: pd.DataFrame) -> set:
    ips = set()
    for col in ['src_ip', 'dest_ip']:
        if col in df.columns:
            ips.update(df[col].dropna().unique())
    return ips

ips1 = get_unique_ips(df1)
ips2 = get_unique_ips(df2)

if ips1 or ips2:
    print(f"\n🌐 Unique IPs in PCAP 1 (first 10): {list(ips1)[:10]}")
    print(f"🌐 Unique IPs in PCAP 2 (first 10): {list(ips2)[:10]}")

print("""
\n💡 IMPACT OF HOME_NET ON DETECTION:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

1. FALSE NEGATIVES from a misconfigured HOME_NET:
   If the PCAP's IPs are not in HOME_NET, rules like:
     alert tcp $EXTERNAL_NET any -> $HOME_NET 445 (msg:"ET EXPLOIT MS17-010")
   will NOT trigger, even if the traffic is malicious.

   → Solution: Set HOME_NET = 'any' for forensic analysis of external
     PCAPs, or specifically include the PCAP's ranges.

2. FALSE POSITIVES with HOME_NET = any:
   If HOME_NET = any, EXTERNAL_NET = !$HOME_NET = empty → no external traffic.
   Many rules detecting external attacks will stop triggering.

   → Optimal solution: Use the PCAP's actual IP range as HOME_NET.

3. PROPOSED ADJUSTMENT FOR THESE PCAPs:
   For offline forensic analysis:

   # To cover all common private + public ranges:
   HOME_NET: "[192.168.0.0/16,10.0.0.0/8,172.16.0.0/12,any]"
   EXTERNAL_NET: "any"

   This maximizes detection in both PCAPs without creating conflicts.

4. PROPOSED RULE ADJUSTMENTS:
   - To reduce false positives in PCAP 1 (web scans):
     * Disable low-severity "Policy" rules
     * Increase the threshold on scan rules: 'threshold: type both, track by_src, count 50, seconds 60'

   - For PCAP 2 (WannaCry):
     * Enable specific rules: et/open category 'emerging-exploit'
     * Turn on alerts for SMBv1: 'proto:smb' in suricata.yaml → smb: enabled: yes
""")

---
## Section 7: Technical Report Template

### Instructions for the student

Based on the evidence generated by the previous sections, complete the technical report below. **Copy the actual excerpts** from `fast.log` and `eve.json` that you found while running the notebook.

---

### ✍️ Assignment Question

> **What differences do you observe between the "activity profile" detected by Suricata in the webserver scans and probes PCAP and the historical WannaCry/EternalBlue PCAP, and what adjustments (configuration and/or rules) would you propose to reduce false positives without losing detection capability?**

---

### 1) Evidence — Alerts and JSON Events

**Instructions:** Copy at least 2 alerts from `fast.log` and 2 events from `eve.json` for each PCAP.

#### PCAP 1: Webserver Scans and Probes

**fast.log alert #1:**
```
[PASTE A fast.log ALERT FROM PCAP 1 HERE]
Example: 11/24/2024-10:23:45.123456  [**] [1:2010935:4] ET SCAN Nmap Scripting Engine [**] [Classification: Web Application Attack] [Priority: 1] {TCP} 192.168.1.100:52341 -> 10.0.0.5:80
```

**fast.log alert #2:**
```
[PASTE ANOTHER fast.log ALERT FROM PCAP 1 HERE]
```

**eve.json event #1:**
```json
[PASTE A JSON EVENT FROM PCAP 1 HERE]
Important fields: timestamp, event_type, src_ip, dest_ip, proto, alert.signature, alert.category
```

**eve.json event #2:**
```json
[PASTE ANOTHER JSON EVENT FROM PCAP 1 HERE]
```

#### PCAP 2: WannaCry / EternalBlue

**fast.log alert #1:**
```
[PASTE A fast.log ALERT FROM PCAP 2 HERE]
Example: 05/18/2017-03:04:05.678901  [**] [1:2023047:5] ET EXPLOIT MS17-010 EternalBlue Exploit [**] [Classification: Attempted Administrator Privilege Gain] [Priority: 1] {TCP} 192.168.1.50:12345 -> 192.168.1.200:445
```

**fast.log alert #2:**
```
[PASTE ANOTHER fast.log ALERT FROM PCAP 2 HERE]
```

**eve.json event #1:**
```json
[PASTE A JSON EVENT FROM PCAP 2 HERE]
```

**eve.json event #2:**
```json
[PASTE ANOTHER JSON EVENT FROM PCAP 2 HERE]
```

---

### 2) Interpretation — Profile Comparison

**Instructions:** Write 3–5 paragraphs comparing the two PCAPs. Use the comparison table from Section 6.

**Guiding questions:**
- Which protocol dominates in each PCAP? Why is that relevant?
- What type of activity predominates: reconnaissance, exploitation, propagation?
- How do you infer this from the `fast.log` signatures and the `eve.json` `event_type` values?
- Which of the two represents a greater immediate risk, and why?

> **[WRITE YOUR INTERPRETATION HERE]**

---

### 3) Configuration — suricata.yaml and network variables

**Instructions:** Connect your analysis to the Suricata configuration.

**Guiding questions:**
- How do `HOME_NET` and `EXTERNAL_NET` affect the alerts you observed?
- Were there alerts you expected to see that didn't appear? What might explain that?
- What change did you make to `rule-files`? Why is it necessary?
- How many rules did `suricata-update` load? Without it, would you have detected WannaCry?
- What `HOME_NET` adjustment do you propose to reduce false positives in PCAP 1?

> **[WRITE YOUR CONFIGURATION ANALYSIS HERE]**

---

### 4) Optimization Proposals

**Concrete proposals based on your analysis:**

| # | Observed problem | Proposed adjustment | Expected impact |
|---|-------------------|-----------------|-----------------|
| 1 | [e.g.: Too many low-priority alerts in PCAP 1] | [e.g.: Disable the "Potentially Bad Traffic" category] | [e.g.: -30% alerts, same critical coverage] |
| 2 | [e.g.: MS17-010 not detected without suricata-update] | [e.g.: Automate daily suricata-update via cron] | [e.g.: Up-to-date coverage against recent threats] |
| 3 | [COMPLETE] | [COMPLETE] | [COMPLETE] |

In [ ]:
print("=" * 70)
print("FINAL PRACTICE SUMMARY")
print("=" * 70)

# Safe file existence checks
try:
    suricata_ok = bool(shutil.which('suricata'))
except:
    suricata_ok = False

try:
    config_exists = CONFIG_FILE.exists()
except PermissionError:
    result = subprocess.run(['sudo', 'test', '-e', str(CONFIG_FILE)], capture_output=True)
    config_exists = result.returncode == 0

config_content = read_file_safe(CONFIG_FILE)
rules_yaml_ok = 'suricata.rules' in config_content if config_content else False

try:
    rules_file_exists = (RULES_DIR / 'suricata.rules').exists()
except PermissionError:
    result = subprocess.run(['sudo', 'test', '-e', str(RULES_DIR / 'suricata.rules')], capture_output=True)
    rules_file_exists = result.returncode == 0

try:
    pcap1_results_ok = (PCAP1_RESULTS / 'fast.log').exists()
except PermissionError:
    result = subprocess.run(['sudo', 'test', '-e', str(PCAP1_RESULTS / 'fast.log')], capture_output=True)
    pcap1_results_ok = result.returncode == 0

try:
    pcap2_results_ok = (PCAP2_RESULTS / 'fast.log').exists()
except PermissionError:
    result = subprocess.run(['sudo', 'test', '-e', str(PCAP2_RESULTS / 'fast.log')], capture_output=True)
    pcap2_results_ok = result.returncode == 0

summary = {
    'Suricata installed':              suricata_ok,
    'suricata.yaml configured':        config_exists,
    'Rule paths adjusted':             rules_yaml_ok,
    'suricata-update run':             rules_file_exists,
    'PCAP 1 analyzed':                 pcap1_results_ok,
    'PCAP 2 analyzed':                 pcap2_results_ok,
    'Comparative analysis completed':  True,
}

rules_count = count_rules(RULES_DIR / 'suricata.rules')

print("\nComponent status:")
print("-" * 70)
for task, done in summary.items():
    status = "OK" if done else "FAILED"
    print(f"{task:40s}: {status}")

print(f"\nRules loaded: {rules_count['active']:,} active")

alerts_1 = len(parse_fast_log(PCAP1_RESULTS / 'fast.log')) if pcap1_results_ok else 0
alerts_2 = len(parse_fast_log(PCAP2_RESULTS / 'fast.log')) if pcap2_results_ok else 0

print("\nAlerts generated:")
print("-" * 70)
print(f"PCAP 1 (Webserver Scans):    {alerts_1:>8} alerts")
print(f"PCAP 2 (WannaCry/EternalBlue): {alerts_2:>8} alerts")
print(f"TOTAL:                        {alerts_1 + alerts_2:>8} alerts")

print(f"\nResults saved to: {RESULTS_DIR}")
print("\nPractice completed. See section 7 for the technical report.")

---
## Section 7: Technical Report Template

### Instructions for the student

Based on the evidence generated by the previous sections, complete the technical report below. **Copy the actual excerpts** from `fast.log` and `eve.json` that you found while running the notebook.

---

### ✍️ Assignment Question

> **What differences do you observe between the "activity profile" detected by Suricata in the webserver scans and probes PCAP and the historical WannaCry/EternalBlue PCAP, and what adjustments (configuration and/or rules) would you propose to reduce false positives without losing detection capability?**

---

### 1) Evidence — Alerts and JSON Events

**Instructions:** Copy at least 2 alerts from `fast.log` and 2 events from `eve.json` for each PCAP.

#### PCAP 1: Webserver Scans and Probes

**fast.log alert #1:**
```
[PASTE A fast.log ALERT FROM PCAP 1 HERE]
Example: 11/24/2024-10:23:45.123456  [**] [1:2010935:4] ET SCAN Nmap Scripting Engine [**] [Classification: Web Application Attack] [Priority: 1] {TCP} 192.168.1.100:52341 -> 10.0.0.5:80
```

**fast.log alert #2:**
```
[PASTE ANOTHER fast.log ALERT FROM PCAP 1 HERE]
```

**eve.json event #1:**
```json
[PASTE A JSON EVENT FROM PCAP 1 HERE]
Important fields: timestamp, event_type, src_ip, dest_ip, proto, alert.signature, alert.category
```

**eve.json event #2:**
```json
[PASTE ANOTHER JSON EVENT FROM PCAP 1 HERE]
```

#### PCAP 2: WannaCry / EternalBlue

**fast.log alert #1:**
```
[PASTE A fast.log ALERT FROM PCAP 2 HERE]
Example: 05/18/2017-03:04:05.678901  [**] [1:2023047:5] ET EXPLOIT MS17-010 EternalBlue Exploit [**] [Classification: Attempted Administrator Privilege Gain] [Priority: 1] {TCP} 192.168.1.50:12345 -> 192.168.1.200:445
```

**fast.log alert #2:**
```
[PASTE ANOTHER fast.log ALERT FROM PCAP 2 HERE]
```

**eve.json event #1:**
```json
[PASTE A JSON EVENT FROM PCAP 2 HERE]
```

**eve.json event #2:**
```json
[PASTE ANOTHER JSON EVENT FROM PCAP 2 HERE]
```

---

### 2) Interpretation — Profile Comparison

**Instructions:** Write 3–5 paragraphs comparing the two PCAPs. Use the comparison table from Section 6.

**Guiding questions:**
- Which protocol dominates in each PCAP? Why is that relevant?
- What type of activity predominates: reconnaissance, exploitation, propagation?
- How do you infer this from the `fast.log` signatures and the `eve.json` `event_type` values?
- Which of the two represents a greater immediate risk, and why?

> **[WRITE YOUR INTERPRETATION HERE]**

---

### 3) Configuration — suricata.yaml and network variables

**Instructions:** Connect your analysis to the Suricata configuration.

**Guiding questions:**
- How do `HOME_NET` and `EXTERNAL_NET` affect the alerts you observed?
- Were there alerts you expected to see that didn't appear? What might explain that?
- What change did you make to `rule-files`? Why is it necessary?
- How many rules did `suricata-update` load? Without it, would you have detected WannaCry?
- What `HOME_NET` adjustment do you propose to reduce false positives in PCAP 1?

> **[WRITE YOUR CONFIGURATION ANALYSIS HERE]**

---

### 4) Optimization Proposals

**Concrete proposals based on your analysis:**

| # | Observed problem | Proposed adjustment | Expected impact |
|---|-------------------|-----------------|-----------------|
| 1 | [e.g.: Too many low-priority alerts in PCAP 1] | [e.g.: Disable the "Potentially Bad Traffic" category] | [e.g.: -30% alerts, same critical coverage] |
| 2 | [e.g.: MS17-010 not detected without suricata-update] | [e.g.: Automate daily suricata-update via cron] | [e.g.: Up-to-date coverage against recent threats] |
| 3 | [COMPLETE] | [COMPLETE] | [COMPLETE] |

In [ ]:
# ── Final practice summary ─────────────────────────────────────────────
print("=" * 70)
print("📋 FINAL PRACTICE SUMMARY")
print("=" * 70)

def _exists(path):
    try:
        return path.exists()
    except PermissionError:
        result = subprocess.run(['sudo', 'test', '-e', str(path)], capture_output=True)
        return result.returncode == 0

summary = {
    'Suricata installed':         bool(shutil.which('suricata')),
    'suricata.yaml reviewed':     _exists(CONFIG_FILE),
    'Rule path adjusted':         'suricata.rules' in read_file_safe(CONFIG_FILE),
    'suricata-update run':        _exists(RULES_DIR / 'suricata.rules'),
    'PCAP 1 analyzed':            _exists(PCAP1_RESULTS / 'fast.log'),
    'PCAP 2 analyzed':            _exists(PCAP2_RESULTS / 'fast.log'),
    'Comparison generated':       True,
}

rules_count = count_rules(RULES_DIR / 'suricata.rules')

for task, done in summary.items():
    icon = '✅' if done else '❌'
    print(f"  {icon} {task}")

print(f"\n📊 Rules loaded: {rules_count['active']:,} active")

alerts_1 = len(parse_fast_log(PCAP1_RESULTS / 'fast.log')) if _exists(PCAP1_RESULTS / 'fast.log') else 0
alerts_2 = len(parse_fast_log(PCAP2_RESULTS / 'fast.log')) if _exists(PCAP2_RESULTS / 'fast.log') else 0

print(f"\n📄 Alerts generated:")
print(f"   PCAP 1 (Webserver Scans):      {alerts_1}")
print(f"   PCAP 2 (WannaCry/EternalBlue): {alerts_2}")

print(f"\n📁 Results saved to: {RESULTS_DIR}")
print("\n✅ Practice completed. Proceed to complete the technical report (Section 7).")